# MyDigitalTwin — Google & YouTube
**Notebook 06 — Ingestion, exploration, nettoyage → Delta Lake**

Sources :
- `Mon activité/Recherche/MonActivité.html` → recherches Google
- `Mon activité/Chrome/MonActivité.html` → historique Chrome
- `YouTube et YouTube Music/historique/watch-history.html` → vidéos YouTube regardées
- `YouTube et YouTube Music/historique/Historique des recherches.html` → recherches YouTube

Outputs :
- `warehouse/google_searches`
- `warehouse/google_chrome`
- `warehouse/youtube_watch`
- `warehouse/youtube_searches`

## Objectifs
- **NLP Clone** : requêtes Google → vocabulaire naturel
- **K-Means** : activité temporelle toutes sources

## 0. Initialisation

In [1]:
import sys, os
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import PROCESSED_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from bs4 import BeautifulSoup
from datetime import datetime, timezone
import re

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Google") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

GOOGLE_ROOT = os.path.join(PROCESSED_DATA, "GOOGLE")

# Mapping mois français → numéro
MOIS = {
    "janv": 1, "févr": 2, "mars": 3, "avr": 4, "mai": 5, "juin": 6,
    "juil": 7, "août": 8, "sept": 9, "oct": 10, "nov": 11, "déc": 12
}

def parse_google_date(text):
    """
    Parse les dates Google Takeout en français.
    Ex: '15 mars 2026, 18:29:40 CET'
        '20 mars 2026, 08:24:55 CET'
    """
    if not text:
        return 0
    m = re.search(
        r'(\d{1,2})\s+(\w+\.?)\s+(\d{4}),?\s+(\d{2}):(\d{2}):(\d{2})',
        text
    )
    if not m:
        return 0
    day, month_str, year, h, mi, s = m.groups()
    month_key = month_str.lower().rstrip('.')
    month_num = MOIS.get(month_key[:4], None)
    if not month_num:
        return 0
    try:
        dt = datetime(int(year), month_num, int(day), int(h), int(mi), int(s),
                      tzinfo=timezone.utc)
        return int(dt.timestamp() * 1000)
    except:
        return 0

def parse_html_activity(html_path, activity_type):
    """
    Parser générique pour les fichiers MonActivité.html Google.
    Retourne une liste de dicts avec : title, url, timestamp_ms, activity_type
    """
    print(f"  Lecture : {html_path.split('/')[-1]}...")
    with open(html_path, encoding="utf-8", errors="replace") as f:
        content = f.read()

    soup  = BeautifulSoup(content, "lxml")
    cells = soup.find_all("div", class_="outer-cell")
    print(f"  Cellules trouvées : {len(cells):,}")

    rows = []
    total_cells = len(cells)
    for idx, cell in enumerate(cells):
        if idx % 10000 == 0 and idx > 0:
            print(f"  ... {idx:,}/{total_cells:,} cellules traitees")
        content_div = cell.find("div", class_="content-cell")
        if not content_div:
            continue

        links    = content_div.find_all("a")
        main_lnk = links[0] if links else None
        sec_lnk  = links[1] if len(links) > 1 else None

        title  = main_lnk.get_text(strip=True) if main_lnk else ""
        url    = main_lnk.get("href", "")[:300] if main_lnk else ""
        author = sec_lnk.get_text(strip=True) if sec_lnk and \
                 any(x in sec_lnk.get("href", "") for x in ["channel", "user", "@"]) else ""

        full_text  = content_div.get_text(separator=" | ")
        ts_ms      = parse_google_date(full_text)

        if not title and not ts_ms:
            continue

        rows.append({
            "title":         title,
            "url":           url,
            "author":        author,
            "timestamp_ms":  ts_ms,
            "activity_type": activity_type,
            "platform":      "google",
        })

    print(f"  Lignes extraites : {len(rows):,}")
    return rows

def make_df(rows, extra_fields=None):
    """Crée un DataFrame PySpark via fichier NDJSON sur volume partagé."""
    import json as _json, os as _os
    _tmp_path = "/opt/spark/data/tmp_make_df.json"
    if _os.path.exists(_tmp_path): _os.remove(_tmp_path)
    with open(_tmp_path, "w", encoding="utf-8") as _f:
        for row in rows:
            print(_json.dumps(row), file=_f)
    print(f"  Fichier temp : {_tmp_path} ({len(rows):,} lignes)")
    df = spark.read.json(_tmp_path)
    df = df \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("char_count",    F.length("title")) \
        .withColumn("word_count",    F.size(F.split(F.trim("title"), r"\s+")))
    return df

print("✓ Fonctions prêtes")

Spark version : 3.5.5
✓ Fonctions prêtes


26/04/26 00:39:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE 1 — Recherches Google
### 1.1 Ingestion

In [2]:
print("[1/4] Lecture Google Searches...")
search_path = f"{GOOGLE_ROOT}/Mon activité/Recherche/MonActivité.html"

search_rows = parse_html_activity(search_path, "google_search")
df_search   = make_df(search_rows)

# Extraire le terme de recherche depuis l'URL
# Ex: https://www.google.fr/search?q=afterwork+bruxelles
df_search = df_search.withColumn(
    "query",
    F.regexp_extract(F.col("url"), r"[?&]q=([^&]+)", 1)
).withColumn(
    "query",
    F.regexp_replace(F.col("query"), r"\+", " ")
)

print(f"\nDataFrame : {df_search.count():,} lignes")
df_search.select("query", "title", "event_date").show(10, truncate=60)
print(f"[1/4] Google Searches OK — {len(search_rows):,} lignes")

[1/4] Lecture Google Searches...
  Lecture : MonActivité.html...
  Cellules trouvées : 55,827
  ... 10,000/55,827 cellules traitees
  ... 20,000/55,827 cellules traitees
  ... 30,000/55,827 cellules traitees
  ... 40,000/55,827 cellules traitees
  ... 50,000/55,827 cellules traitees
  Lignes extraites : 55,827
  Fichier temp : /opt/spark/data/tmp_make_df.json (55,827 lignes)



DataFrame : 55,827 lignes
+------------------------------------------------------------+------------------------------------------------------------+-------------------+
|                                                       query|                                                       title|         event_date|
+------------------------------------------------------------+------------------------------------------------------------+-------------------+
|https://www.themoviedb.org/tv/124364-from/cast%3Flanguage...|FROM (TV Series 2022- ) - Distribution des rôles et équip...|2026-04-24 20:18:22|
|          https://en.wikipedia.org/wiki/From_%28TV_series%29|                                From (TV series) - Wikipedia|2026-04-24 20:18:10|
|           https://www.imdb.com/title/tt9813792/fullcredits/|           From (TV Series 2022– ) - Full cast & crew - IMDb|2026-04-24 20:18:02|
|                                   from distributions series|                                   from distrib

### 1.2 Exploration

In [3]:
print("=== Recherches par année ===")
df_search.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_search.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top 20 mots recherchés ===")
df_search.filter(F.col("query") != "") \
    .withColumn("word", F.explode(F.split(F.lower("query"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

print("\n=== Top 20 requêtes complètes ===")
df_search.filter(F.col("query") != "") \
    .groupBy("query").count() \
    .orderBy(F.desc("count")).limit(20).show(truncate=50)

=== Recherches par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2016|  129|
|      2017|   99|
|      2018| 1399|
|      2019| 5561|
|      2020| 5074|
|      2021| 8132|
|      2022|10374|
|      2023| 9514|
|      2024|10742|
|      2025| 4666|
|      2026|  137|
+----------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0| 2678|
|         1| 1665|
|         2|  559|
|         3|  326|
|         4|   78|
|         5|   27|
|         6|  166|
|         7|  342|
|         8|  948|
|         9| 1553|
|        10| 2030|
|        11| 2369|
|        12| 2714|
|        13| 2549|
|        14| 3134|
|        15| 3422|
|        16| 3146|
|        17| 3804|
|        18| 4049|
|        19| 3960|
+----------+-----+
only showing top 20 rows


=== Top 20 mots recherchés ===


+---------+-----+
|     word|count|
+---------+-----+
|      des|  360|
|      les|  354|
| belgique|  309|
|      une|  291|
|    faire|  287|
|      one|  268|
|      sur|  268|
|  youtube|  262|
|     fifa|  261|
|   google|  240|
|streaming|  229|
|     pour|  216|
|    piece|  208|
|minecraft|  204|
|      the|  197|
|   %c3%a0|  189|
|  comment|  185|
|     plus|  184|
|     avec|  183|
|     prix|  178|
+---------+-----+


=== Top 20 requêtes complètes ===
+--------------------------------+-----+
|                           query|count|
+--------------------------------+-----+
|                         youtube|   87|
|https://www.dreamland.be/e/fr/dl|   69|
|                          amazon|   54|
|                            hdss|   48|
|                            sncb|   44|
|                          krefel|   40|
|      https://tickets.imagix.be/|   39|
|                   google flight|   36|
|                       dreamland|   33|
|                        stg gege|   33|

### 1.3 Écriture Parquet

In [4]:
print("[1/4] Ecriture Delta google_searches...")
df_search.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "google_searches"))
print(f"google_searches -- {df_search.count():,} lignes")
print("[1/4] google_searches DONE")

[1/4] Ecriture Delta google_searches...


google_searches -- 55,827 lignes
[1/4] google_searches DONE


---
## PARTIE 2 — Historique Chrome
### 2.1 Ingestion

In [5]:
print("[2/4] Lecture Chrome history...")
chrome_path = f"{GOOGLE_ROOT}/Mon activité/Chrome/MonActivité.html"

chrome_rows = parse_html_activity(chrome_path, "chrome_visit")
df_chrome   = make_df(chrome_rows)

# Extraire le domaine visité
df_chrome = df_chrome.withColumn(
    "domain",
    F.regexp_extract(F.col("url"), r"https?://(?:www\.)?([^/]+)", 1)
)

print(f"\nDataFrame : {df_chrome.count():,} lignes")
df_chrome.select("title", "domain", "event_date").show(10, truncate=60)
print(f"[2/4] Chrome OK — {len(chrome_rows):,} lignes")

[2/4] Lecture Chrome history...
  Lecture : MonActivité.html...
  Cellules trouvées : 338
  Lignes extraites : 338
  Fichier temp : /opt/spark/data/tmp_make_df.json (338 lignes)

DataFrame : 338 lignes
+------------------------------+----------+-------------------+
|                         title|    domain|         event_date|
+------------------------------+----------+-------------------+
|                              |          |2025-12-29 23:31:38|
|Photopea | Online Photo Editor|google.com|2025-12-24 00:13:40|
|Photopea | Online Photo Editor|google.com|2025-12-13 21:37:36|
|Photopea | Online Photo Editor|google.com|2025-12-10 22:36:30|
|Photopea | Online Photo Editor|google.com|2025-12-01 00:50:54|
|Photopea | Online Photo Editor|google.com|2025-11-30 22:35:34|
|Photopea | Online Photo Editor|google.com|2025-11-28 00:13:46|
|Photopea | Online Photo Editor|google.com|2025-11-15 20:50:27|
|Photopea | Online Photo Editor|google.com|2025-11-07 18:01:31|
|Photopea | Online Photo Edito

### 2.2 Exploration

In [6]:
print("=== Visites par année ===")
df_chrome.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 20 domaines visités ===")
df_chrome.filter(F.col("domain") != "") \
    .groupBy("domain").count() \
    .orderBy(F.desc("count")).limit(20).show(truncate=50)

print("\n=== Activité par heure ===")
df_chrome.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_hour").count().orderBy("event_hour").show()

=== Visites par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2025|  338|
+----------+-----+


=== Top 20 domaines visités ===
+-------------------------+-----+
|                   domain|count|
+-------------------------+-----+
|               google.com|  294|
|          mail.google.com|   24|
|                google.be|    8|
|              youtube.com|    5|
|chromewebstore.google.com|    3|
|      accounts.google.com|    2|
+-------------------------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|   94|
|         1|    8|
|         2|    1|
|        11|    1|
|        13|   42|
|        14|    6|
|        16|    3|
|        17|    1|
|        18|    1|
|        19|    3|
|        20|    3|
|        21|   74|
|        22|   40|
|        23|   61|
+----------+-----+



### 2.3 Écriture Parquet

In [7]:
print("[2/4] Ecriture Delta google_chrome...")
df_chrome.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "google_chrome"))
print(f"google_chrome -- {df_chrome.count():,} lignes")
print("[2/4] google_chrome DONE")

[2/4] Ecriture Delta google_chrome...
google_chrome -- 338 lignes
[2/4] google_chrome DONE


---
## PARTIE 3 — YouTube Watch History
### 3.1 Ingestion

In [ ]:
print("[3/4] Lecture YouTube Watch history...")
yt_watch_path = f"{GOOGLE_ROOT}/YouTube et YouTube Music/historique/watch-history.html"

yt_watch_rows = parse_html_activity(yt_watch_path, "youtube_watch")
df_yt_watch   = make_df(yt_watch_rows)

# Extraire l'ID vidéo YouTube
df_yt_watch = df_yt_watch \
    .withColumn(
        "video_id",
        F.regexp_extract(F.col("url"), r"[?&]v=([a-zA-Z0-9_-]{11})", 1)
    )

print(f"\nDataFrame : {df_yt_watch.count():,} lignes")
df_yt_watch.select("title", "author", "video_id", "event_date").show(10, truncate=50)
print(f"[3/4] YouTube Watch OK — {len(yt_watch_rows):,} lignes")

### Filtrage des pubs

In [3]:
# Filtrer les publicités
df_yt_watch = df_yt_watch.filter(
    # Exclure les URLs publicitaires
    ~F.col("url").contains("googleadservices") &
    ~F.col("url").contains("doubleclick") &
    ~F.col("url").contains("googlevideo.com/videoplayback") &
    # Exclure les titres génériques de pubs
    ~F.col("title").rlike(r"(?i)^(publicité|pub|advertisement|ad)$") &
    # Garder uniquement les vraies vidéos YouTube (avec video_id)
    (F.col("video_id") != "")
)

print(f"Après filtrage pubs : {df_yt_watch.count():,} lignes")

Après filtrage pubs : 14,076 lignes


### 3.2 Exploration

In [4]:
print("=== Vidéos regardées par année ===")
df_yt_watch.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 15 chaînes les plus regardées ===")
df_yt_watch.filter(F.col("author") != "") \
    .groupBy("author").count() \
    .orderBy(F.desc("count")).limit(15).show(truncate=50)

print("\n=== Top 15 vidéos les plus regardées ===")
df_yt_watch.filter(F.col("title") != "") \
    .groupBy("title", "author").count() \
    .orderBy(F.desc("count")).limit(15).show(truncate=50)

print("\n=== Activité par heure ===")
df_yt_watch.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top 20 mots dans les titres ===")
df_yt_watch.filter(F.col("title") != "") \
    .withColumn("word", F.explode(F.split(F.lower("title"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

=== Vidéos regardées par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2023| 2118|
|      2024| 4592|
|      2025| 5502|
|      2026| 1864|
+----------+-----+


=== Top 15 chaînes les plus regardées ===
+-----------------------+-----+
|                 author|count|
+-----------------------+-----+
|               SQUEEZIE|  171|
|             Squa' Sh*t|  137|
|Squeezie - Rediffusions|  129|
|        SQUEEZIE GAMING|   81|
|                  AlaGT|   77|
|               Anyme TV|   70|
|                  Mastu|   65|
|           DJ LONY (BE)|   65|
|                Arsenal|   64|
|           Zen Emission|   62|
|          Lacrimosophia|   54|
|          DJ ZAYON PROD|   53|
|                  HOUDI|   47|
|        TravisScottVEVO|   45|
|                Inoxtag|   45|
+-----------------------+-----+


=== Top 15 vidéos les plus regardées ===
+--------------------------------------------------+----------+-----+
|                                             ti

### 3.3 Écriture Parquet

In [5]:
print("[3/4] Ecriture Delta youtube_watch...")
df_yt_watch.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "youtube_watch"))
print(f"youtube_watch -- {df_yt_watch.count():,} lignes")
print("[3/4] youtube_watch DONE")

[3/4] Ecriture Delta youtube_watch...


youtube_watch -- 14,076 lignes
[3/4] youtube_watch DONE


---
## PARTIE 4 — Recherches YouTube
### 4.1 Ingestion

In [6]:
print("[4/4] Lecture YouTube Search history...")
yt_search_path = f"{GOOGLE_ROOT}/YouTube et YouTube Music/historique/Historique des recherches.html"

yt_search_rows = parse_html_activity(yt_search_path, "youtube_search")
df_yt_search   = make_df(yt_search_rows)

print(f"\nDataFrame : {df_yt_search.count():,} lignes")
df_yt_search.select("title", "event_date").show(10, truncate=60)
print(f"[4/4] YouTube Search OK")

[4/4] Lecture YouTube Search history...
  Lecture : Historique des recherches.html...
  Cellules trouvées : 6,047
  Lignes extraites : 6,047
  Fichier temp : /opt/spark/data/tmp_make_df.json (6,047 lignes)

DataFrame : 6,047 lignes
+------------------------------------------------------------+-------------------+
|                                                       title|         event_date|
+------------------------------------------------------------+-------------------+
|         [BEL STH] Pokémon TCG: Mega Evolution—Perfect Order|2026-04-24 14:49:36|
|                                       CELSIUS - Live fit go|2026-04-24 14:49:30|
|    Square: (NL) Driving Test Routes App Explainer (QR Code)|2026-04-24 14:44:14|
|                                         NIVEA BLACK & WHITE|2026-04-24 14:44:09|
|        adnova_creative_46396ff270474e299208f52170523e9e.mp4|2026-04-24 14:35:50|
|INH DAILYBANKING UNDER18 a433163d Kids&TeensCardInHandBat...|2026-04-24 14:32:19|
|NOUVEAU CeraVe Aqua 

In [7]:
df_yt_search = df_yt_search.filter(
    # Exclure les titres de pubs (patterns reconnaissables)
    ~F.col("title").rlike(r"(?i)^Shortened:") &
    ~F.col("title").rlike(r"(?i)^(RAD|ROK|INH|AYF|ADS|ADV)[\s_]") &
    ~F.col("title").rlike(r"[0-9]{4}x[0-9]{3,4}") &  # dimensions vidéo ex: 1920x1080
    ~F.col("title").rlike(r"_[0-9]+s$") &              # durée en secondes ex: _30s
    ~F.col("title").rlike(r"(?i)(partnership|campaign|creative)") &
    # Garder uniquement les vraies recherches (au moins 2 mots ou titre court naturel)
    (F.length("title") > 2) &
    ~F.col("title").rlike(r"^[A-Z0-9_\s]{20,}$")  # codes en majuscules = pubs
)

print(f"Après filtrage pubs : {df_yt_search.count():,} lignes")
df_yt_search.select("title", "event_date").show(10, truncate=60)

Après filtrage pubs : 5,148 lignes
+------------------------------------------------------------+-------------------+
|                                                       title|         event_date|
+------------------------------------------------------------+-------------------+
|         [BEL STH] Pokémon TCG: Mega Evolution—Perfect Order|2026-04-24 14:49:36|
|                                       CELSIUS - Live fit go|2026-04-24 14:49:30|
|    Square: (NL) Driving Test Routes App Explainer (QR Code)|2026-04-24 14:44:14|
|                                         NIVEA BLACK & WHITE|2026-04-24 14:44:09|
|NOUVEAU CeraVe Aqua - Gel Hydratant Acide hyaluronique, d...|2026-04-24 14:32:13|
|    260415 BE10 P00004306 KUL Infodagen Adaptatie 16x9 L 003|2026-04-24 14:21:11|
|               BEFR - STAND UP - MISSED OPPORTUNITIES - 6SEC|2026-04-24 14:21:05|
|                                  #1 Driving Test Routes App|2026-04-24 14:06:11|
|                              S.Pellegrino. Bring y

### 4.2 Exploration

In [8]:
print("=== Recherches YouTube par année ===")
df_yt_search.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 20 termes recherchés ===")
df_yt_search.groupBy("title").count() \
    .orderBy(F.desc("count")).limit(20).show(truncate=60)

print("\n=== Top 20 mots ===")
df_yt_search.filter(F.col("title") != "") \
    .withColumn("word", F.explode(F.split(F.lower("title"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

=== Recherches YouTube par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2023|  663|
|      2024| 1774|
|      2025| 2157|
|      2026|  554|
+----------+-----+


=== Top 20 termes recherchés ===
+------------------------------------------------------------+-----+
|                                                       title|count|
+------------------------------------------------------------+-----+
|                                           Choisissez Chrome|   39|
|                          BNPPF_HY4 Boost_9x16_FR_12sec_2505|   26|
|                 https://www.youtube.com/watch?v=7CEm-f2YvIQ|   19|
|                                       CFC英语口播 0202 横版|   19|
|DEWALT® UK | On the jobsite, leaders aren't born. They're...|   18|
|INFMAC DAILYBANKING BUDGETING  RhodeMacro AV 16x9 6s B1 N...|   12|
|                                   Le pack ING Do More 18-25|   11|
|           Changez d'avis sur le reconditionné - Back Market|   11|
|FINAL 15123076 FY23Q4 

### 4.3 Écriture Parquet

In [9]:
print("[4/4] Ecriture Delta youtube_searches...")
df_yt_search.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "youtube_searches"))
print(f"youtube_searches -- {df_yt_search.count():,} lignes")
print("[4/4] youtube_searches DONE")

[4/4] Ecriture Delta youtube_searches...


youtube_searches -- 5,148 lignes
[4/4] youtube_searches DONE


In [11]:
spark.stop()